In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

In [3]:
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

data_path = ROOT / "data" / "processed" / "ca1_sales.parquet"

df = pd.read_parquet(data_path)

print("Shape:", df.shape)
display(df.head())

Shape: (5832737, 15)


,id,item_id,dept_id,cat_id,store_id,d,sales,date,wm_yr_wk,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,sell_price
0,FOODS_1_001_CA_1_validation,FOODS_1_001,FOODS_1,FOODS,CA_1,d_1,3,2011-01-29,11101,NaN,NaN,NaN,NaN,0,2.0
1,FOODS_1_001_CA_1_validation,FOODS_1_001,FOODS_1,FOODS,CA_1,d_2,0,2011-01-30,11101,NaN,NaN,NaN,NaN,0,2.0
2,FOODS_1_001_CA_1_validation,FOODS_1_001,FOODS_1,FOODS,CA_1,d_3,0,2011-01-31,11101,NaN,NaN,NaN,NaN,0,2.0
3,FOODS_1_001_CA_1_validation,FOODS_1_001,FOODS_1,FOODS,CA_1,d_4,1,2011-02-01,11101,NaN,NaN,NaN,NaN,1,2.0
4,FOODS_1_001_CA_1_validation,FOODS_1_001,FOODS_1,FOODS,CA_1,d_5,4,2011-02-02,11101,NaN,NaN,NaN,NaN,1,2.0


In [4]:
print(df.dtypes)

id                         str
item_id                    str
dept_id                    str
cat_id                     str
store_id                   str
d                          str
sales                    int64
date            datetime64[us]
wm_yr_wk                 int64
event_name_1               str
event_type_1               str
event_name_2               str
event_type_2               str
snap_CA                  int64
sell_price             float64
dtype: object


In [5]:
print("Rows:", len(df))
print("Products:", df["item_id"].nunique())
print("Date range:", df["date"].min(), "to", df["date"].max())
print("Duplicate product-date pairs:",
      df.duplicated(["item_id", "date"]).sum())
print("Missing sales:", df["sales"].isna().sum())

Rows: 5832737
Products: 3049
Date range: 2011-01-29 00:00:00 to 2016-04-24 00:00:00
Duplicate product-date pairs: 0
Missing sales: 0


In [6]:
# Create calendar features
df["day_of_week"] = df["date"].dt.dayofweek
df["day_of_month"] = df["date"].dt.day
df["month"] = df["date"].dt.month
df["year"] = df["date"].dt.year
df["week_of_year"] = df["date"].dt.isocalendar().week.astype("int16")

In [7]:
# Create lag features
for lag in [7, 14, 28]:
    df[f"lag_{lag}"] = (df.groupby("item_id")["sales"].shift(lag))

In [8]:
# Create rolling mean features
grouped_sales = df.groupby("item_id")["sales"]

df["rolling_mean_7"] = grouped_sales.transform(
    lambda s: s.shift(1).rolling(window=7, min_periods=7).mean()
)

df["rolling_mean_28"] = grouped_sales.transform(
    lambda s: s.shift(1).rolling(window=28, min_periods=28).mean()
)

In [9]:
feature_cols = ["day_of_week", "day_of_month", "month", "year", "week_of_year",
                "lag_7", "lag_14", "lag_28", "rolling_mean_7", "rolling_mean_28"]

df[["item_id", "date", "sales"] + feature_cols].head(35)

,item_id,date,sales,day_of_week,day_of_month,month,year,week_of_year,lag_7,lag_14,lag_28,rolling_mean_7,rolling_mean_28
0,FOODS_1_001,2011-01-29,3,5,29,1,2011,4,NaN,NaN,NaN,NaN,NaN
1,FOODS_1_001,2011-01-30,0,6,30,1,2011,4,NaN,NaN,NaN,NaN,NaN
2,FOODS_1_001,2011-01-31,0,0,31,1,2011,5,NaN,NaN,NaN,NaN,NaN
3,FOODS_1_001,2011-02-01,1,1,1,2,2011,5,NaN,NaN,NaN,NaN,NaN
4,FOODS_1_001,2011-02-02,4,2,2,2,2011,5,NaN,NaN,NaN,NaN,NaN
5,FOODS_1_001,2011-02-03,2,3,3,2,2011,5,NaN,NaN,NaN,NaN,NaN
6,FOODS_1_001,2011-02-04,0,4,4,2,2011,5,NaN,NaN,NaN,NaN,NaN
7,FOODS_1_001,2011-02-05,2,5,5,2,2011,5,3.0,NaN,NaN,1.428571,NaN
8,FOODS_1_001,2011-02-06,0,6,6,2,2011,5,0.0,NaN,NaN,1.285714,NaN
9,FOODS_1_001,2011-02-07,0,0,7,2,2011,6,0.0,NaN,NaN,1.285714,NaN


In [10]:
print(df[feature_cols].isna().sum())

day_of_week            0
day_of_month           0
month                  0
year                   0
week_of_year           0
lag_7              21343
lag_14             42686
lag_28             85372
rolling_mean_7     21343
rolling_mean_28    85372
dtype: int64


In [11]:
output_path = ROOT / "data" / "processed" / "ca1_features.parquet"

df.to_parquet(output_path, index=False)

print("Saved:", output_path)
print("Shape:", df.shape)

Saved: /home/ajaya-kumar-patel/Desktop/Placement/Project/retail-demand-forecasting/data/processed/ca1_features.parquet
Shape: (5832737, 25)
